<a href="https://colab.research.google.com/github/anaishinschberger-cloud/my_llm/blob/main/01_bigram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**BIGRAM**

1 : Imports

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Model and training configuration
batch_size = 32 # Number of independent sequences processed at once
context_length = 8 # Maximum number of context tokens used for prediction
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
torch.manual_seed(1337)

In [3]:
!wget -q https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt -O input.txt

2 : Read text data

In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

3 : Encode

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Map each character to an integer and define the reverse mapping
char_to_id = { ch:i for i,ch in enumerate(chars) }
id_to_char = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [char_to_id[c] for c in s] # Convert a string into a sequence of integer IDs
decode = lambda l: ''.join([id_to_char[i] for i in l]) # Reconstruct a string from a sequence of integer IDs

# Store the encoded text in a tensor before splitting the data
data = torch.tensor(encode(text), dtype=torch.long)

3 : Prepare data

In [6]:
# Use 90% of the data for training and the remainder for validation
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# Build batches of sequences with targets shifted one character ahead
def get_batch(split):
    data = train_data if split == 'train' else val_data
    start_indices = torch.randint(len(data) - context_length, (batch_size,))
    inputs = torch.stack([data[i:i+context_length] for i in start_indices])
    targets = torch.stack([data[i+1:i+context_length+1] for i in start_indices])
    inputs, targets = inputs.to(device), targets.to(device)
    return inputs, targets

4: Evaluate

In [7]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

5 : Create bigram class

In [8]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Look up the next-token logits associated with each input token
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):


        logits = self.token_embedding_table(idx) # Dimensions: batch (B), sequence length (T), vocabulary (C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx holds the current context token IDs with shape (B, T)
        for _ in range(max_new_tokens):
            # Compute logits for the current sequence
            logits, loss = self(idx)
            # Keep the logits at the final sequence position
            logits = logits[:, -1, :] # Resulting tensor shape: (B, C)
            # Convert logits into probabilities using softmax
            probs = F.softmax(logits, dim=-1) # Dimensions: batch (B), vocabulary (C)
            # Draw the next token according to the predicted probabilities
            idx_next = torch.multinomial(probs, num_samples=1) # One sampled token ID per sequence: (B, 1)
            # Extend each sequence with its sampled token ID
            idx = torch.cat((idx, idx_next), dim=1) # Sequence length increases by one: (B, T+1)
        return idx


6 : Train

In [9]:

model = BigramLanguageModel(vocab_size)
m = model.to(device)

# Set up AdamW to update the model parameters
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Run the training loop
for iter in range(max_iters):

    # Periodically measure loss on the training and validation data
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Draw a batch from the training data
    input_batch, target_batch = get_batch('train')

    # Compute the loss before updating the parameters
    logits, loss = model(input_batch, target_batch)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Produce text with the trained model
initial_context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(initial_context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.7305, val loss 4.7241
step 300: train loss 2.8110, val loss 2.8249
step 600: train loss 2.5434, val loss 2.5682
step 900: train loss 2.4932, val loss 2.5088
step 1200: train loss 2.4863, val loss 2.5035
step 1500: train loss 2.4665, val loss 2.4921
step 1800: train loss 2.4683, val loss 2.4936
step 2100: train loss 2.4696, val loss 2.4846
step 2400: train loss 2.4638, val loss 2.4879
step 2700: train loss 2.4738, val loss 2.4911

MARI he avayokis erceller thour d, myono thishe me tord se by he me, Forder anen: at trselorinjulour t yoru thrd wo ththathy IUShe bavidelanoby man ond be jus as g e atot Meste hrle s, ppat t JLENCOLIUS:
Oppid tes d s o ged moer y pevehear soue maramapay fo t: bueyo malalyo!
Duir.
Fl ke it I t l o'ddre d ondu s?
cr, havetrathackes w.
PUpee meshancun, hrendspouthoulouren whel's'sesoread pe, s whure our heredinsethes; sedsend r lo pamit,
QUMIVIVIOfe m ne RDINid we tr ort; t:
MINENXI l dintandore r
